### 微调

In [1]:
import torch
import pandas as pd
import random
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import TaskType, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING
import datetime
from IPython.display import HTML, display

# 设置 CUDA 内存分配器
torch.cuda.set_per_process_memory_fraction(0.9)  # 使用 90% 的可用 GPU 内存
torch.cuda.empty_cache()

# 打印 PyTorch 配置和 GPU 属性
print(torch.__config__.show(), torch.cuda.get_device_properties(0))

# 定义全局变量和参数
model_name_or_path = '/root/dataDisk/hf/hub/models/chatglm3-6b'
train_data_paths = [
    'data/zhouyi_dataset_20240118_152413.csv',
    'data/zhouyi_dataset_20240118_163659.csv',
    'data/zhouyi_dataset_handmade.csv'
]
seed = 8
max_input_length = 512
max_output_length = 1536
prompt_text = ''
lora_rank = 8  # 减小 LoRA 秩
lora_alpha = 16
lora_dropout = 0.05

# 加载多个数据集并合并
datasets = [load_dataset("csv", data_files=path)['train'] for path in train_data_paths]
dataset = concatenate_datasets(datasets)
column_names = dataset.column_names

print(f"Combined dataset size: {len(dataset)}")

# 显示数据集的随机元素
def show_random_elements(dataset, num_examples=5):
    picks = random.sample(range(len(dataset)), num_examples)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

show_random_elements(dataset)

# 初始化 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, revision='b098244')

# 编写数据处理函数
def tokenize_func(example):
    question = prompt_text + example['content']
    if 'input' in example and example['input'].strip():
        question += f'\n{example["input"]}'
    answer = example['summary']
    q_ids = tokenizer.encode(question, add_special_tokens=False, truncation=True, max_length=max_input_length-2)
    a_ids = tokenizer.encode(answer, add_special_tokens=False, truncation=True, max_length=max_output_length-1)
    
    input_ids = tokenizer.build_inputs_with_special_tokens(q_ids, a_ids)
    question_length = len(q_ids) + 2
    labels = [-100] * question_length + input_ids[question_length:]
    
    return {'input_ids': input_ids, 'labels': labels}

# 处理数据集
tokenized_dataset = dataset.map(tokenize_func, remove_columns=column_names, num_proc=4)
tokenized_dataset = tokenized_dataset.shuffle(seed=seed)

# 定义数据整理器
class DataCollatorForChatGLM:
    def __init__(self, tokenizer, max_length=2048):
        self.pad_token_id = tokenizer.pad_token_id
        self.max_length = max_length

    def __call__(self, batch_data):
        max_len = min(max(len(x['input_ids']) for x in batch_data), self.max_length)
        input_ids = torch.full((len(batch_data), max_len), self.pad_token_id, dtype=torch.long)
        labels = torch.full((len(batch_data), max_len), -100, dtype=torch.long)
        for i, item in enumerate(batch_data):
            input_len = min(len(item['input_ids']), max_len)
            input_ids[i, :input_len] = torch.tensor(item['input_ids'][:input_len])
            labels[i, :input_len] = torch.tensor(item['labels'][:input_len])
        return {'input_ids': input_ids, 'labels': labels}

# 准备数据整理器
data_collator = DataCollatorForChatGLM(tokenizer)

# QLoRA 量化配置
q_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 加载预训练模型
model = AutoModel.from_pretrained(
    model_name_or_path,
    quantization_config=q_config,
    device_map='auto',
    trust_remote_code=True,
    revision='b098244',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# 准备 LoRA 训练
model = prepare_model_for_kbit_training(model)
target_modules = TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']

lora_config = LoraConfig(
    target_modules=target_modules,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias='none',
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 配置训练参数
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"models/{model_name_or_path.split('/')[-1]}-epoch3-{timestamp}"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,  # 减小批量大小
    gradient_accumulation_steps=4,  # 增加梯度累积步数
    learning_rate=1e-4,  # 降低学习率
    num_train_epochs=10,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,  # 启用梯度检查点
    max_grad_norm=0.3,  # 添加梯度裁剪
    dataloader_num_workers=2,  # 增加数据加载器的工作进程数
)

# 初始化训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 开始训练
trainer.train()

# 保存训练好的模型
trainer.model.save_pretrained(output_dir)

/root/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.3.6 (Git Hash 86e6af5974177e513fd3fee58425e1063e7f1361)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code=sm_70;-gencode;arch=compute_75,code=sm_75;-gencode;arch=compute_80,code=sm_80;-gencode;arch=compute_86,code=sm_86;-gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_90,code=sm_90
  - CuDNN 8.7
  - Magma 2.6.1
  - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, CUDA_VERSION=11.8, CUDNN_VERSION=8.7.0, CXX_COMPILER=/opt/rh/devtoolset-9/root/usr/bin/c++, CXX_FLAGS= -D_GLIBCXX_USE_CXX11_ABI=0 -fabi-version=11 -fvisibility-inlines-hidden -DUS

,content,summary
0,师卦涉及哪些哲学思想？,在周易中，师卦是一个由坎卦（水）和坤卦（地）相叠而成的异卦。这一卦象代表着军队的力量和军情的总指挥，预示着吉祥无灾。象辞中描述了地中有水的情景，寓意着君子应当像大地一样容纳和畜养大众。师卦的解释强调选择德高望重的长者来统率军队，才能获得吉祥无咎。另外，师卦也象征着困难重重，需要包容别人、艰苦努力，及时行事，严于律已。在事业、经商、求名、婚恋等方面的决策中，都需要警惕潜在敌人，小心谨慎，合作与决断兼顾，方能成功。
1,周易中屯卦的解释是什么？,在周易中，屯卦是一个大吉大利的卦象，预示着吉祥和大利。然而，不利于出门，但有利于建国封侯。屯卦由上卦坎（水）下卦震（雷）组成，坎为云，震为雷。预示着云行雷动的卦象。君子观此卦象，取法于云雷，用云的恩泽，雷的威严来治理国事。屯卦象征着开始困难，需要毅力和果敢才能获得吉利。身处困境需要多加辛苦努力，排除困难，方可通达，有初难后解之象。因此，对于事业创业而言，应当小心翼翼，勇往直前，灵活机动，可望获得大的成功。但也需注意到仍有困难存在，务必有他人相助，平时应多施恩惠。对于经商，起初多有挫折，必须坚定信念，积极进取，行动果断，若仍无法摆脱困境，则应退守保全，等待机会，再展宏图。对于婚恋，好事多磨，忠贞纯洁，大胆追求，能够成功。屯卦的核心哲学在于，初难后解，需要毅力和坚忍不拔的毅力和锲而不舍的奋斗精神，但也需得到贤德之人的帮助才能摆脱困境。
2,周易的坤卦讲述了什么？,坤卦，是周易中的一卦，由两个坤卦叠加而成，代表大地的顺从和承载。在这个卦中，预示着大吉大利，预言了雌马牵动的吉兆，君子出行会先迷失，后来找到主人，有利的方向是西南，不利的方向是东北。总体上，这是一个吉利的卦象。《象辞》中讲到，大地形势平和，君子观卦以厚德载物。坤卦的解释中提到，坤卦代表柔顺和地气舒展之象，主张妥善安排，等待时机，宜顺从运势以制定大事。在传统解卦中，坤卦代表谨慎行事，灵活适应，依循正道获得吉利。在事业、经商、婚姻和决策等方面，坤卦均主张顺从自然规律，勿急进，谋求长远利益。
3,"""乾卦和启蒙教育之间有何联系？","""乾卦""\nsummary: ""《易经》中的乾卦是六十四卦中的首卦，象征天，由六个阳爻组成，代表着刚健强劲的特性。其卦辞为“元、亨、利、贞”，预示着吉祥如意，同时也教导人们遵守天道的德行。乾卦所蕴含的核心哲学是：天道刚健，运行不已，君子观此卦象，从而以天为法，自强不息。""\n\ncomment: ""在传统解卦中，乾卦预示着大吉大利，事业如日中天，但也提醒要警惕盛极必衰的道理。经商方面顺利发展，但要冷静分析形势，坚持商业道德。对于婚恋，尽管阳盛阴衰，但刚柔可相济，最终形成美满结果。总体而言，乾卦代表着充满活力和力量的时机，但也需要保持谦逊谨慎的态度，以应对可能出现的困难。"
4,周易中的蒙卦含义是什么？,蒙卦是由艮卦（山）下，坎卦（水）上组成的异卦相叠。它代表着通泰，启蒙的意义。在这里，卜者并非是在向幼稚愚昧的人取求，而是幼稚愚昧的人在向卜者求教。第一次卜筮就得到了神灵的指示。然而，如果轻慢不敬地再三卜筮的话，神灵便不会再示警。总的来说，这是一个吉利的卜问。\n\n蒙卦的核心在于山下有泉的形象，寓意着启蒙。君子观此卦象，应当以果敢坚毅的行动来培养自身的品德，像山泉一样果断行动。然而，此卦乃是离宫四世卦，它代表着回还往复、疑惑不前、多忧愁过失，因而属于凶卦。\n\n蒙卦在个人发展、事业经商、求名婚恋等方面的解释不一。在事业方面，表示事业初建，具有启蒙和通达之象，需要勇敢坚毅的行动；而在经商方面，需要务必小心谨慎，树立高尚的商业道德，不可急功近利；求名方面，需要接受良好的基础教育，陶冶情操。整体而言，此卦提示须忍耐待机而动，听取别人意见，方能通达运势。


Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100%|██████████| 7/7 [00:18<00:00,  2.65s/it]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


trainable params: 1,949,696 || all params: 6,245,533,696 || trainable%: 0.031217444255383614


Step,Training Loss
10,4.042700
20,4.139000
30,3.808100
40,3.487600
50,2.942000
60,3.025400
70,2.621300
80,2.511300
90,2.209900
100,1.862400


### 完整的 inference 对比代码

In [6]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

# 模型ID或本地路径
model_name_or_path = '/root/dataDisk/hf/hub/models/chatglm3-6b'

# 设置映射计算数据类型
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])

# 加载量化后的基础模型
base_model = AutoModel.from_pretrained(model_name_or_path,
                                       quantization_config=q_config,
                                       device_map='auto',
                                       trust_remote_code=True,
                                       revision='b098244')
base_model.requires_grad_(False)
base_model.eval()

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

# 加载微调后的模型
epochs = 3
timestamp = "20240707_044048"
peft_model_path = "models/chatglm3-6b-epoch3-20240707_044048"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag=f"ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-{timestamp}"

# 定义比较函数
def compare_chatglm_results(query, base_model, qlora_model, training_tag):
    base_response, base_history = base_model.chat(tokenizer, query=query)
    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = qlora_model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\n微调后（{training_tag}）：\n{ft_response}")
    return base_response, ft_response

# 进行对比
base_response, ft_response = compare_chatglm_results("解释下乾卦是什么？", base_model, qlora_model, training_tag)
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model, training_tag)
base_response, ft_response = compare_chatglm_results("师卦是什么？", base_model, qlora_model, training_tag)

Loading checkpoint shards: 100%|██████████| 7/7 [00:04<00:00,  1.71it/s]


问题：解释下乾卦是什么？

原始输出：
乾卦是周易中的第一卦，由两个乾卦叠加而成，象征着天，它是天地初始、万物生长的象征。在卦象上，乾卦是由两个乾卦叠加而成，显示了天行健健的特征。在卦象中，乾卦 represented as a tree, with its branches reaching towards the sky, demonstrating the dynamic and strong growth of the tree.

在卦象中，乾卦代表天，象征着天行健健，这一卦象预示了天地初始、万物生长的情景。在运势上，乾卦预示着顺利、刚健、健全，预示了顺利、顺利达成目标、 robust、健全等方面。

在事业方面，乾卦预示着事业成功，预示着事业顺利、险象环生，要谨慎小心。在经商方面，乾卦预示着经商顺利，但也需谨慎处理商事。在婚恋方面，乾卦预示着婚姻顺利，但也需谨慎选择配偶。

在决策方面，乾卦预示着天行健健，预示了顺利、刚健、健全，要求健锐、坚毅、 robust，三思而后行。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-20240707_044048）：
[gMASK] sop 解释下乾卦是什么？ 在周易中，乾卦是六十四卦之首，由六个阳爻组成，象征着天。它所代表的是刚健、健行、刚健不屈的意境。乾卦的核心哲学是：天道刚健，运行不已，君子观此卦象，从而以天为法，自强不息。

乾卦象征天，为大通而至正。得此卦者，名利双收，应把握机会，争取成果。然而，切勿过于骄傲自满，而应保持谦逊、冷静和警惕。在事业、经商、求名等方面，乾卦皆暗示着大吉大利，但也警示着必须坚持正道、修养德行，方能永远亨通。

在婚恋方面，乾卦提示着阳盛阴衰，但也强调刚柔相济，相互补足，形成美满的结果。在决策方面，则是强调刚健、正直、公允，自强不息的实质，需要修养德行、坚定信念，方能克服困难，消除灾难。
问题：周易中的讼卦是什么

原始输出：
讼卦是周易中的一卦，由两个卦相交而形成。其中乾卦代表天，象征刚健震动；坤卦代表地，象征顺和忍让。讼卦的卦象是：下乾上坤。

讼卦的卦辞认为：天行刚健，君子观此卦象，应当以忍让和缓释争端。具体来说，讼卦预示着否定的形势，然而希望获得成功，需要谨慎行事，包容他人，以及积极行动。

讼卦分析卦象，天行刚健，